# Residential pricing optimization using ML (Machine learning)

## Objetive

The main goal of this project is to develop machine learning models to **predict and optimize housing prices** based on various factors, such as leads, visits, and reservations. By leveraging models like **XGBoost**, **RandomForest**, and **time series forecasting**, this project aims to support data-driven pricing decisions, prevent unprofitable pricing strategies, and ensure optimal price points for residential properties.


## Business Understanding

## Problem

In the real estate industry, pricing decisions often rely on intuition or simple rule-based approaches, which can lead to suboptimal pricing and reduced profitability. This project addresses the following key challenges:

1. **Accurate Price Prediction**: Determining the most accurate price for a property based on relevant business metrics.
2. **Price Optimization**: Preventing prices from falling outside the optimal range that maximizes profitability.
3. **Data-Driven Decisions**: Shifting from intuition-based pricing to data-driven insights for better decision-making.

The core problem is to create a reliable model that can estimate the ideal price for a property based on the observed data and business metrics. Additionally, the model should help in identifying pricing anomalies that could affect profitability.

## Methodology

This project follows the **CRISP-DM** framework to ensure a structured approach to data mining:

1. **Business Understanding**: Understanding the key business variables (leads, visits, reservations) and the target variable (price).
2. **Data Understanding**: Exploring and preprocessing the data to ensure its quality and suitability for modeling.
3. **Data Preparation**: Cleaning, transforming, and preparing the data, including feature engineering and encoding categorical variables.
4. **Modeling**: Training multiple machine learning models (Random Forest, XGBoost, and possibly time-series forecasting models) to predict housing prices.
5. **Evaluation**: Evaluating the performance of each model using metrics such as **RMSE**, **MAE**, and **R²**.
6. **Deployment**: If applicable, preparing the model for deployment (though this step will not be included in the current scope).

## Results Expected

The expected outcome of this project is the creation of a robust machine learning model that can:

- **Accurately predict housing prices** based on key features like leads, visits, and reservations.
- **Provide insights** on which features are most important for determining prices, enabling better decision-making.
- **Optimize pricing** to ensure that it aligns with market conditions and business goals, avoiding prices that are too high or too low.

Ultimately, the goal is to create a tool that helps real estate professionals set prices based on data, improving profitability and reducing pricing errors.

## Models Used

- **Random Forest**: A versatile ensemble learning method that will provide a good baseline for understanding feature importance and predicting prices.
- **XGBoost**: A more advanced boosting model known for its performance in regression tasks, which will be used to fine-tune predictions.
- **Time Series Forecasting** (if applicable): A forecasting approach to predict future prices based on past data.

---

In [24]:
pip install xgboost

   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 1.3/150.0 MB 11.3 MB/s eta 0:00:14
   - -------------------------------------- 6.3/150.0 MB 19.3 MB/s eta 0:00:08
   -- ------------------------------------- 11.0/150.0 MB 20.9 MB/s eta 0:00:07
   ---- ----------------------------------- 18.6/150.0 MB 25.0 MB/s eta 0:00:06
   ----- ---------------------------------- 21.5/150.0 MB 22.3 MB/s eta 0:00:06
   ------ --------------------------------- 23.1/150.0 MB 20.6 MB/s eta 0:00:07
   ------- -------------------------------- 28.3/150.0 MB 20.6 MB/s eta 0:00:06
   -------- ------------------------------- 33.6/150.0 MB 21.3 MB/s eta 0:00:06
   --------- ------------------------------ 37.5/150.0 MB 20.9 MB/s eta 0:00:06
   ----------- ---------------------------- 43.8/150.0 MB 21.9 MB/s eta 0:00:05
   ------------ --------------------------- 48.0/150.0 MB 21.7 MB/s eta 0:00:05
   ------------- -------------------------- 52.4/15

In [28]:
# Import the required libraries
# Standard Libraries
import pandas as pd
import numpy as np
import joblib

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Model libraries
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from xgboost import XGBRegressor

# Accuracy and evaluation
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Plotting
import seaborn as sns
import matplotlib.pyplot as plt

# Enviroment
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [30]:
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['figure.dpi'] = 100

## Data understanding

### Data overview

In [36]:
# Get the dataframe from /data
data = pd.read_csv('data/Weekly.csv')

In [40]:
data.head()

,ID,Fecha,Semana,Proyecto,Modelo,Website Leads,Showroom Visits,Modelo Tipo,Separaciones,Price,Inventario restante,Ventas,Conversión Separaciones (%),Conversión Ventas (%),Inventario Disponible,Precio Promedio del Proyecto-Modelo
0,01/01/2024CBR-BR2,01/01/2024,1,CBR,BR2,0,0,Normal,0,0,0,NO,0.000000,NaN,NaN,NaN
1,01/01/2024CBR-COR3,01/01/2024,1,CBR,COR3,28,1,Normal,0,0,0,NO,0.000000,NaN,NaN,NaN
2,01/01/2024CBR-VIS2,01/01/2024,1,CBR,VIS2,50,7,Normal,3,"2,550,000.00",0,SI,0.428571,NaN,NaN,NaN
3,01/01/2024ALS-IB-3N,01/01/2024,1,ALS,IB-3N,20,0,Normal,0,0,0,NO,0.000000,NaN,NaN,NaN
4,01/01/2024ALS-IB-6,01/01/2024,1,ALS,IB-6,34,6,Normal,1,"2,760,000.00",0,SI,NaN,NaN,NaN,NaN


In [50]:
data.columns

Index(['ID', 'Fecha', 'Semana', 'Proyecto', 'Modelo', 'Website Leads',
       'Showroom Visits', 'Modelo Tipo', 'Separaciones', 'Price',
       'Inventario restante', 'Ventas', 'Conversión Separaciones (%)',
       'Conversión Ventas (%)', 'Inventario Disponible',
       'Precio Promedio del Proyecto-Modelo'],
      dtype='object')

In [52]:
data.dtypes

ID                                      object
Fecha                                   object
Semana                                   int64
Proyecto                                object
Modelo                                  object
Website Leads                            int64
Showroom Visits                          int64
Modelo Tipo                             object
Separaciones                             int64
Price                                   object
Inventario restante                      int64
Ventas                                  object
Conversión Separaciones (%)            float64
Conversión Ventas (%)                  float64
Inventario Disponible                  float64
Precio Promedio del Proyecto-Modelo    float64
dtype: object

- 1197 Rows
- Columns:
  - ID: Unique ID from system ERP
  - Fecha: First day of the week
  - Semana: Week from year
  - Proyecto: Name of proyect inside the enterprise
  - Modelo: Type of residential/housing that contains the data
  - Website lead: Are the potentials customers or visitors
  - Showroom visits: Number of visit to check the residential/housing presential
  - Separaciones: ** Number of books after of showroom stage
  - Price: **It´s the target** and is the actual price of housing (The number 0 is the values that not are ready)
  - Inventario restante: Its the stock of residentials or number of available residentials/housings
  - Conversión Separaciones (%): It´s the number of percent when the customers passed from 'Showroom stage' -> 'Separaciones'
  - Conversión Ventas (%): It´s the number of percent when the customers passed from 'Separaciones' -> 'Ventas'
  - Inventario Disponible: Number of residentials in stock
  - Precio Promedio del Proyecto-Modelo: Its the average of price by model

In [42]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 16 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   ID                                   1197 non-null   object 
 1   Fecha                                1197 non-null   object 
 2   Semana                               1197 non-null   int64  
 3   Proyecto                             1197 non-null   object 
 4   Modelo                               1197 non-null   object 
 5   Website Leads                        1197 non-null   int64  
 6   Showroom Visits                      1197 non-null   int64  
 7   Modelo Tipo                          1197 non-null   object 
 8   Separaciones                         1197 non-null   int64  
 9   Price                                1197 non-null   object 
 10  Inventario restante                  1197 non-null   int64  
 11  Ventas                        